# Basic operator construction

This notebook shows the shortest EDKit workflow: define local terms, specify where they act, construct an `Operator`, and compare its linear-map, dense, and sparse forms.


In [ ]:
using LinearAlgebra
DEV = true

function find_local_edkit(start = pwd())
    dir = abspath(start)
    while true
        candidate = joinpath(dir, "src", "EDKit.jl")
        isfile(candidate) && return candidate
        parent = dirname(dir)
        parent == dir && error("Could not locate src/EDKit.jl from $(start)")
        dir = parent
    end
end

if DEV
    include(find_local_edkit())
    using .EDKit
else
    using EDKit
end
using SparseArrays


In [ ]:
L = 6
delta = 1.1

mats = [
    fill(spin("XX"), L - 1);
    fill(spin("YY"), L - 1);
    fill(delta * spin("ZZ"), L - 1);
]

inds = vcat(
    [[i, i + 1] for i in 1:L-1],
    [[i, i + 1] for i in 1:L-1],
    [[i, i + 1] for i in 1:L-1],
)

H = operator(mats, inds, L)


In [ ]:
psi = randn(ComplexF64, 2^L) |> normalize
Hpsi_linear = H * psi
H_dense = Array(H)
H_sparse = sparse(H)

summary = (; 
    system_size = L,
    dimension = size(H, 1),
    nterms = length(H),
    linear_vs_dense_error = norm(Hpsi_linear - H_dense * psi),
    dense_vs_sparse_error = norm(H_dense - Matrix(H_sparse)),
    lowest_three_eigs = eigvals(Hermitian(H_dense))[1:3],
)

summary
